In [1]:
import torch
from model_utils import load_transformer_model, load_prompt_embedding_model_and_tokenizer, get_reference_audio_latent, \
    load_encoder, load_silence_latent, decode_latent_and_save_audio

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model_dtype = torch.bfloat16
# model_repo = "ACE-Step/acestep-v15-base" if torch.cuda.is_available() else "ACE-Step/acestep-v15-turbo-shift1"
model_repo = "ACE-Step/acestep-v15-turbo-shift1"
llm_repo_id = "ACE-Step/acestep-5Hz-lm-0.6B"

cuda


In [3]:
silence_latent = load_silence_latent(model_repo, "silence_latent.pt", device, model_dtype)

In [4]:
vae = load_encoder("./models/ace-step-vae/config.json", "./models/ace-step-vae/checkpoint.ckpt", device, model_dtype)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\clip\clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention


W0530 11:56:17.230000 22504 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [5]:
model = load_transformer_model(model_repo, model_dtype, device)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:454: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:639: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\finite_scalar_quantization.py:159: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\lookup_free_quantization.py:244: FutureWarning: `torch.cuda.amp.autocast(args...)

In [6]:
# Conditioning on reference audio does not currently work
refer_audio_acoustic_hidden_states_packed, refer_audio_order_mask = get_reference_audio_latent(vae, device, model_dtype, silence_latent)

torch.Size([2, 2880000])


In [7]:
prompt_embedding_model, tokenizer = load_prompt_embedding_model_and_tokenizer(llm_repo_id, model_dtype, device)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\z00489tu\.cache\huggingface\hub\models--ACE-Step--acestep-5Hz-lm-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling

In [9]:
# Conditioning on prompt does not currently work
max_length = 2048
prompt = "heavy metal deftones guitars distorted"
text_input_dict = tokenizer(prompt, padding="longest", truncation=True, max_length=2048, return_tensors="pt",)
prompt_token_ids = text_input_dict["input_ids"].to(device)
text_attention_mask = text_input_dict.attention_mask.bool()

tokens = prompt_embedding_model(prompt_token_ids, padding="longest", return_tensors="pt")
text_hidden_states = tokens['last_hidden_state'].to(device).to(model_dtype)
print(text_hidden_states.shape)

torch.Size([1, 6, 1024])


In [10]:
# Conditioning on prompt does not currently work, override with zeros
text_hidden_states = torch.zeros(1, 77, 1024, dtype=model_dtype, device=device)
text_attention_mask = torch.zeros(text_hidden_states.shape[0], text_hidden_states.shape[1], dtype=torch.bool, device=device)
lyric_hidden_states = torch.zeros(1, 123, 1024, dtype=model_dtype, device=device)
lyric_attention_mask = torch.zeros(1, 123, dtype=torch.bool, device=device)

is_covers = torch.Tensor([False]).to(device)

seconds = 60
infer_steps = 50
frames_per_second = 25

seq_len = int(seconds * frames_per_second)

cur_chunk_mask = torch.ones(1, seq_len, 64, dtype=torch.bool, device=device)
cur_src_latents = silence_latent[:, :, :seq_len].permute(0, 2, 1)

print(lyric_hidden_states.shape)
print(lyric_attention_mask.shape)
print(text_hidden_states.shape)
print(text_attention_mask.shape)
print(cur_chunk_mask.shape)
print(cur_src_latents.shape)
print(refer_audio_acoustic_hidden_states_packed.shape)
print(refer_audio_order_mask.shape)

torch.Size([1, 123, 1024])
torch.Size([1, 123])
torch.Size([1, 77, 1024])
torch.Size([1, 77])
torch.Size([1, 1500, 64])
torch.Size([1, 1500, 64])
torch.Size([1, 1500, 64])
torch.Size([1])


In [11]:
outputs = model.generate_audio(
    text_hidden_states=text_hidden_states,
    text_attention_mask=text_attention_mask,
    lyric_hidden_states=lyric_hidden_states,
    lyric_attention_mask=lyric_attention_mask,
    refer_audio_acoustic_hidden_states_packed=refer_audio_acoustic_hidden_states_packed,
    refer_audio_order_mask=refer_audio_order_mask,
    src_latents=cur_src_latents,
    chunk_masks=cur_chunk_mask,
    infer_steps=infer_steps,
    is_covers=is_covers,
    silence_latent=silence_latent,
    use_progress_bar=True,
    shift=1.0,

    repainting_start=torch.tensor([1.0]),
    repainting_end=torch.tensor([0.0]),
    audio_cover_strength=1.0,
    use_repainting=False
)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\residual_fsq.py:170: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled = False):


In [12]:
output_latents = outputs['target_latents'].transpose(1, 2).contiguous()
decode_latent_and_save_audio(output_latents, vae, "output")

torch.Size([1, 2, 2880000])
(2880000, 2)
